# 05 — Serving bundle handoff

This notebook exports the selected Guesser ViT-Tiny checkpoint as a hashed serving bundle and verifies the artifact contract. It is packaging and parity work, not recognition evaluation. The fixed ten-stage evaluator is not implemented; do not report recognition scores from this notebook.

The export logic lives in `models/guesser/training/export_bundle.py`. This notebook calls it with `uv run` because the notebook kernel belongs to the separate Python 3.12 evaluation project. Do not import training modules into this kernel.


## Launch and development workflow

From the repository root, sync the model environment, then launch JupyterLab through the evaluation environment with the repository as its contents root:

```sh
uv sync --project models/guesser --frozen
uv run --project models/guesser/evaluation jupyter lab --ServerApp.root_dir=.
```

The model scripts are the source of truth. Export is opt-in and disabled by default. The export reads the selected checkpoint and `labels.json`; it never downloads data, reads the held-out test split, or modifies the original checkpoint/selection files.


In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
from guesser_evaluation import notebook_tools

NOTEBOOK_TOOLS = notebook_tools.module_source_path("notebook_tools.py")
REPOSITORY_ROOT = NOTEBOOK_TOOLS.parents[5]
MODEL_PROJECT = REPOSITORY_ROOT / "models" / "guesser"
if not (MODEL_PROJECT / "training" / "export_bundle.py").is_file():
    raise FileNotFoundError(f"Bundle exporter not found under {MODEL_PROJECT}")

print("Repository:", REPOSITORY_ROOT)
print("Model project:", MODEL_PROJECT)


## Options

The defaults target the completed production A/B run. A is the provisional default because it won complete-image validation; B is exported for comparison. Neither score is a partial-sketch or gameplay result. Set `RUN_EXPORT = True` only after reviewing the paths.


In [ ]:
# Keep execution disabled until the options and output paths have been reviewed.
RUN_EXPORT = True
CHECKPOINT_ROOT = MODEL_PROJECT / "checkpoints" / "notebook-production-full"
BUNDLE_ROOT = CHECKPOINT_ROOT / "bundles"
RUNS = ("a", "b")
MODEL_IDS = {"a": "guesser-vit-a-v1", "b": "guesser-vit-b-v1"}
PARITY_SAMPLES = 8
OVERWRITE = False

if not RUNS or any(run not in {"a", "b"} for run in RUNS):
    raise ValueError("RUNS must contain 'a' and/or 'b'.")
if PARITY_SAMPLES < 0:
    raise ValueError("PARITY_SAMPLES must be non-negative.")

print("Export enabled:", RUN_EXPORT)
print("Checkpoint root:", CHECKPOINT_ROOT)
print("Bundle root:", BUNDLE_ROOT)
print("Runs / model IDs:", {run: MODEL_IDS[run] for run in RUNS})
print("Parity samples:", PARITY_SAMPLES)
print("Overwrite:", OVERWRITE)


## Readiness check

Confirm the selected checkpoint, selection record, and production `labels.json` are present. This is a read-only check; it does not load the model.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import subprocess

MODEL_ENV = dict(os.environ)
MODEL_ENV.pop("PYTHONPATH", None)
MODEL_ENV.pop("PYTHONHOME", None)
MODEL_ENV["PYTHONUNBUFFERED"] = "1"

RUN_DIRS = {"a": "A_complete_only", "b": "B_partial"}
LABELS_JSON = MODEL_PROJECT / "data" / ".local" / "data" / "production" / "labels.json"
if not LABELS_JSON.is_file():
    raise FileNotFoundError(f"Production labels.json not found: {LABELS_JSON}")
labels_sha256 = hashlib.sha256(LABELS_JSON.read_bytes()).hexdigest()
print("labels.json:", LABELS_JSON)
print("labels sha256:", labels_sha256)

for run in RUNS:
    run_dir = CHECKPOINT_ROOT / RUN_DIRS[run]
    selection_path = run_dir / "checkpoint-selection.json"
    if not selection_path.is_file():
        raise FileNotFoundError(f"Selection record not found: {selection_path}")
    selection = json.loads(selection_path.read_text(encoding="utf-8"))
    checkpoint = Path(selection["best_model_path"])
    if not checkpoint.is_file():
        raise FileNotFoundError(f"Selected checkpoint not found: {checkpoint}")
    if selection.get("labels_sha256") != labels_sha256:
        raise ValueError(f"{run}: labels hash does not match the selection record")
    print(
        run,
        "checkpoint=", checkpoint.name,
        "best_model_score=", selection.get("best_model_score"),
        "preprocessing=", selection.get("preprocess_candidate"),
    )


## Export the bundles

When enabled, this cell invokes `training.export_bundle` for each run. The exporter writes a staging directory, verifies hashes/labels/selection and fixed-tensor parity, then publishes the bundle atomically. If a bundle already exists, the exporter reports `skipped-existing` and leaves it untouched; set `OVERWRITE = True` to replace it.


> **Expected cross-project warning:** `uv` may report that the evaluation `VIRTUAL_ENV` does not match the model or `apps/ai` project environment and will be ignored. This is intentional: the notebook kernel uses Python 3.12, while each command selects its own project with `--project`. Do not add `--active`, which would force the notebook environment. This exact mismatch warning is normal; other errors or nonzero exits still indicate a problem.

In [ ]:
if not RUN_EXPORT:
    print("Export disabled. Review the options, then set RUN_EXPORT = False.")
else:
    for run in RUNS:
        command = [
            "uv", "run", "--project", str(MODEL_PROJECT), "--frozen", "--no-sync",
            "python", "-m", "training.export_bundle",
            "--checkpoint-root", str(CHECKPOINT_ROOT),
            "--run", run,
            "--model-id", MODEL_IDS[run],
            "--output-dir", str(BUNDLE_ROOT),
            "--parity-samples", str(PARITY_SAMPLES),
        ]
        if OVERWRITE:
            command.append("--overwrite")
        print("Launching:", " ".join(command), flush=True)
        if not notebook_tools.run_interruptible_command(command, cwd=MODEL_PROJECT, env=MODEL_ENV):
            raise RuntimeError(f"Export interrupted for run {run}")


## Verify the published bundles

This independent check re-reads each manifest, re-hashes every file, and confirms the label order and selection hash. It does not load the model.


In [ ]:
if not RUN_EXPORT:
    print("Export disabled; nothing to verify.")
else:
    for run in RUNS:
        bundle = BUNDLE_ROOT / MODEL_IDS[run]
        manifest = json.loads((bundle / "manifest.json").read_text(encoding="utf-8"))
        files = manifest["files"]
        assert set(files) == {
            "weights.safetensors", "labels.txt", "labels.json", "checkpoint-selection.json"
        }
        for name, digest in files.items():
            actual = hashlib.sha256((bundle / name).read_bytes()).hexdigest()
            assert actual == digest, (bundle, name)
        labels = (bundle / "labels.txt").read_text(encoding="utf-8").splitlines()
        labels_json = json.loads((bundle / "labels.json").read_text(encoding="utf-8"))
        assert len(labels) == 345
        assert [entry["source_name"] for entry in labels_json] == labels
        selection = json.loads((bundle / "checkpoint-selection.json").read_text(encoding="utf-8"))
        assert selection["labels_sha256"] == files["labels.json"]
        assert selection["best_model_path"] == "weights.safetensors"
        print(
            run,
            "modelId=", manifest["modelId"],
            "revision=", manifest["artifactRevision"],
            "weights=", files["weights.safetensors"][:12],
            "labels=", files["labels.json"][:12],
            "size=", (bundle / "weights.safetensors").stat().st_size,
        )


## Optional AI-service readiness check

This loads the real bundle through `ai_service.bundles.load_bundle` in the `apps/ai` environment. It may install CPU PyTorch for the `vit` extra, so it is disabled by default. Run it when preparing a deployment host.


In [ ]:
RUN_AI_READINESS = True

if not RUN_AI_READINESS:
    print("AI readiness check disabled.")
elif not RUN_EXPORT:
    print("Export disabled; nothing to load.")
else:
    ai_project = REPOSITORY_ROOT / "apps" / "ai"
    readiness_env = dict(MODEL_ENV)
    readiness_env["PYTHONPATH"] = "../.."
    for run in RUNS:
        code = (
            "from pathlib import Path\n"
            "from ai_service.bundles import load_bundle\n"
            f"root = Path({str(BUNDLE_ROOT)!r})\n"
            f"bundle = load_bundle('guesser', {MODEL_IDS[run]!r}, root)\n"
            "print(bundle.readiness())\n"
        )
        subprocess.run(
            [
                "uv", "run", "--project", str(ai_project), "--frozen",
                "--extra", "vit", "python", "-c", code,
            ],
            cwd=ai_project,
            env=readiness_env,
            check=True,
        )


## Next steps

- Copy the chosen bundle directory to the deployment artifact root, for example `/artifacts/<model-id>/`.
- Set `AI_RUNTIME=vit`, `GUESSER_MODEL_ID=<model-id>`, and `AI_SMOKE_TEST=0` on the inference service.
- Run the two-player smoke game and record observed latency and availability.
- Run the ten-stage/partial-sketch evaluation separately before choosing A or B or claiming recognition performance.
- Do not commit the bundle; `checkpoints/` is ignored.
